# M5.0 · 负结果异构实验 —— 朴素的 CPU 卸载

**学习目标：** 用*物理实验*证明为什么把 8B 文本编码器从 GPU 挪到 CPU PyTorch 可能**降低**吞吐：CPU 编码要花几分钟，GPU 全程空转。这个负结果用真实工作负载（在你主机上的有界试点）来教，绝不只用玩具模拟器——凡是用到模拟器的地方都标注 `SIMULATED`。

**先预测：** GPU 文本编码器 warm 约 0.55 秒完成一次编码。同样任务在 16 个 CPU 核上的 PyTorch 需要多久？写下你的估计（冻结实验数据说要差两个数量级——在你主机上验证）。

参考讲义：[M05 讲义](../course/M05_parallel_heterogeneous/LESSON.zh-CN.md) · [English](./05_cpu_offload_negative.ipynb)

In [ ]:
from pathlib import Path
import json
import statistics
import sys
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "lab/reference_runs.json").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open Jupyter from the course repository root (see notebooks/README.md).")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
COURSE_MODE = "REFERENCE"  # <- edit once: REFERENCE | GPU_CLASSROOM | GPU_RESEARCH
RUN_OWN_GPU_ARMS = True   # <- False in GPU modes: reuse the latest physical run (shown run_id)
from lab import notebook_runtime as rt
from lab import notebook_support as ns
session = rt.begin(COURSE_MODE, notebook="05")

def _stamp(kind: str, value: str) -> None:
    """Print a visible origin tag so no number is mistaken for another kind."""
    print(f"[{kind}] {value}")

MEASURED_TODAY = "measured on THIS GPU in THIS session (physical)"
HISTORICAL   = "frozen reference evidence from the committed campaign"
ESTIMATE     = "analytical estimate or simulation - NOT a measurement"

def gpu_cell(fn):
    """Run a physical exercise only in GPU modes; print why otherwise."""
    if session.mode != "REFERENCE":
        return fn()
    print("skipped physical exercise: COURSE_MODE=REFERENCE (no GPU work launched)")
    return None

def want_own_arms():
    """Whether this session executes its own bounded arms (True unless the
    student opted to reuse the latest physical run to save GPU time)."""
    if session.mode == "REFERENCE" or RUN_OWN_GPU_ARMS:
        return True
    print("RUN_OWN_GPU_ARMS=False: reusing the latest physical campaign run(s) "
          "instead of launching new ones (metered-GPU mode)")
    return False

## 1 · 冻结参考案例（GPU vs PyTorch-CPU 编码器）
已提交的洁净室研究在同一主机上测量了同样的冻结提示词：GPU PyTorch 与 CPU PyTorch 文本编码器。查看每提示词 warm 中位数与比值。

In [ ]:
from lab import notebook_support as ns
import statistics

enc = ns.read_json("runs/m51-20261105-cleanroom-01/performance/encoder_ab.json")
g = enc["arms"]["G_PyTorch_GPU"]["prompts"]
c0 = enc["arms"]["C0_PyTorch_CPU"]["prompts"]
for pid in ("p1-zh-scene", "p2-en-scene"):
    gm = g[pid]["warm_median_s"]
    cm = c0[pid]["warm_median_s"]
    print(f"{pid:14s} GPU {gm:7.3f}s   CPU {cm:8.2f}s   CPU/GPU = {cm/gm:7.1f}x")
_stamp(HISTORICAL, "frozen cleanroom campaign: PyTorch CPU TE is ~2 orders "
      "of magnitude slower than GPU TE on the same host")

## 2 · 你自己的有界物理试点（GPU 模式）
每个设备一次隔离探针、一个提示词、一次 warm 重复——这是**有界试点**，明确声明不是完整协议结论。CPU 臂可能要花几分钟；这个“慢”本身就是测量结果。

In [ ]:
def run_te_pilots():
    gpu = session.run_m51_baseline("cuda", repeats=1)
    cpu = session.run_m51_baseline("cpu", repeats=1)
    return gpu, cpu

pilots = gpu_cell(run_te_pilots)
own = None
if pilots is not None:
    import json as _j
    gpr = _j.loads((pilots[0] / "probe.json").read_text())
    cpr = _j.loads((pilots[1] / "probe.json").read_text())
    gp = next(iter(gpr["prompts"].values()))
    cp = next(iter(cpr["prompts"].values()))
    own = {"gpu_warm_s": gp["warm_samples_s"][0],
           "cpu_warm_s": cp["warm_samples_s"][0],
           "cpu_rss_mb": cpr.get("rss_peak_mb"),
           "gpu_rss_mb": gpr.get("rss_peak_mb")}
    _stamp(MEASURED_TODAY,
           f"pilot: GPU TE {own['gpu_warm_s']:.2f}s vs CPU TE "
           f"{own['cpu_warm_s']:.1f}s -> {own['cpu_warm_s']/own['gpu_warm_s']:.0f}x "
           f"slower (n=1, bounded pilot, single prompt)")

## 3 · 流水线视角 —— GPU 在哪里空转
H1 在 ComfyUI 内部运行 CPU 编码器（集成臂）；H2 用四种请求比较顺序执行（GPU TE）与流水线执行（CPU TE 与 GPU 去噪重叠）。下面消费资格认证实验的记录（GPU 模式取最新；REFERENCE 展示历史图景）。

In [ ]:
def latest_record(pattern, fname):
    cands = sorted((ROOT / "runs").glob(f"{pattern}/{fname}"))
    if not cands:
        return None, None
    import json as _j
    return _j.loads(cands[-1].read_text()), cands[-1].parent.name

h1, h1_id = gpu_cell(lambda: latest_record("*H1-cpu-te", "H1_RESULTS.json")) or (None, None)
if h1 is not None and "integrated" in h1:
    session.note_consumed(h1_id, why="current-campaign H1 record")
    for tag, row in h1["integrated"].items():
        print(f"H1 integrated {tag:9s} total={row.get('total_s')}s "
              f"te={row.get('text_encode_s')}s vram_peak={row.get('vram_peak_mb')}MB")
    _stamp(MEASURED_TODAY, "H1 integrated CPU-TE arm from the current campaign")
else:
    print("no fresh H1 record on this host; the frozen study is section 1")

h2, h2_id = gpu_cell(lambda: latest_record("*H2-*", "H2_RESULTS.json")) or (None, None)
if h2 is not None:
    session.note_consumed(h2_id, why="current-campaign H2 record")
    for k, v in h2.items():
        if isinstance(v, (int, float)):
            print(f"H2 {k}: {v}")
    _stamp(MEASURED_TODAY, "H2 sequential-vs-pipelined record from this campaign")
else:
    print("no fresh H2 record; pipelining recovery is taught from the frozen study")

## 4 · 调度器模拟 —— 明确标注
下面的甘特图式模拟演示：当 CPU 编码器足够快（M5.1 的 OpenVINO）时，重叠为什么有效。其数字是**模拟值**，绝非实测。

In [ ]:
_stamp(ESTIMATE, "SIMULATED pipeline schedule - illustrative only")

gpu_t = own["gpu_warm_s"] if own else 0.55     # TE on GPU (s)
dit_t = 12.0                                   # DiT+VAE per request (s, order of M1/B0)
cpu_t = own["cpu_warm_s"] if own else 120.0    # TE on CPU (s)
seq_gpu = 4 * (gpu_t + dit_t)                  # 4 requests, GPU TE
seq_cpu = 4 * (cpu_t + dit_t)                  # 4 requests, naive CPU offload
pipelined = max(4 * cpu_t, gpu_t + 4 * dit_t)  # CPU TE overlapped with GPU work
print(f"SIMULATED 4-request wall: GPU-TE {seq_gpu:.1f}s | naive CPU offload "
      f"{seq_cpu:.1f}s | overlapped {pipelined:.1f}s")
print("naive offload LOSES unless CPU encode overlaps GPU denoise")

## 5 · 动手练习
1. 用你的试点数字计算：重叠设计需要多少 CPU worker 才能完全隐藏 CPU 编码时间？
2. 即使在延迟上输了，为什么 CPU 卸载在 **VRAM** 上仍然划算？什么时候这是正确的权衡？
3. 举一个理由：你这台主机的 CPU/GPU 比值为什么会与冻结实验的两个数量级不同？

In [ ]:
summary = {
    "mode": session.mode,
    "notebook": session.notebook,
    "executions": [e.get("run_id") or e.get("kind") for e in session.executions],
    "consumed": [c.get("run_id") for c in session.consumed],
    "note": "append-only session record; physical evidence lives under runs/",
}
session.finish(summary)

## 验收标准

- [ ] 你能指出哪些数字是**今天在这块 GPU 上实测的**、哪些是
      **历史参考值**、哪些是**估计/模拟**。
- [ ] 你产生的每次物理运行都在 `runs/` 下有可见的 `run_id`。
- [ ] 你没有把 smoke/课堂样本冒充为正式协议结果。
- [ ] 你能解释你报告的每项测量的一个局限性。

## 复现与贡献

```bash
python -m lab doctor                                  # 准入证据 -> runs/doctor.json
python -m lab run --experiment H1 --date <your-tag>  # 完整研究协议
python -m lab report --scope student --kind all       # 只汇总你自己的 runs
```

提交你的 `runs/<run-id>/*.json` 记录（绝不写 `evidence/`），并在 PR 中说明：
假设 → 协议 → 数字 → 局限。欢迎负结果——测得的退化是科学结果，不是失败。

```bash
python -m lab run --experiment H2 --date <your-tag>  # pipelined arm
```